In [1]:

!pip install ucimlrepo


In [2]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:

adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

df = pd.concat([X, y], axis=1)

print("Dataset downloaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())


In [4]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

# Download dataset
adult = fetch_ucirepo(id=2)

# Separate features and target
X = adult.data.features
y = adult.data.targets

# Combine into one DataFrame
df = pd.concat([X, y], axis=1)

print("Dataset downloaded successfully!")
print("Original shape:", df.shape)

display(df.head())

Dataset downloaded successfully!
Original shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [6]:
# Select 5,000 random records
df = df.sample(n=5000, random_state=42)

# Reset index
df = df.reset_index(drop=True)

print("Dataset Shape:", df.shape)

Dataset Shape: (5000, 15)


In [7]:
df.to_csv("adult_raw.csv", index=False)

print("Dataset saved successfully!")


Dataset saved successfully!


In [8]:

# Check dataset dimensions
print("Number of Rows and Columns:", df.shape)

# Display all column names
print("\nColumn Names:")
print(df.columns.tolist())

# Display first 5 records
print("\nFirst 5 Records:")
display(df.head())

# Check dataset information
print("\nDataset Information:")
df.info()


Number of Rows and Columns: (5000, 15)

Column Names:
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

First 5 Records:


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,41,Private,53956,Bachelors,13,Divorced,Sales,Not-in-family,White,Male,0,1980,56,United-States,<=50K
1,46,Private,431515,Bachelors,13,Never-married,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K.
2,45,Private,185385,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,<=50K
3,54,Private,83103,Bachelors,13,Married-civ-spouse,Sales,Husband,White,Male,0,0,67,United-States,<=50K.
4,38,Private,219902,HS-grad,9,Separated,Transport-moving,Unmarried,Black,Female,0,0,30,United-States,<=50K



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             5000 non-null   int64 
 1   workclass       4903 non-null   object
 2   fnlwgt          5000 non-null   int64 
 3   education       5000 non-null   object
 4   education-num   5000 non-null   int64 
 5   marital-status  5000 non-null   object
 6   occupation      4903 non-null   object
 7   relationship    5000 non-null   object
 8   race            5000 non-null   object
 9   sex             5000 non-null   object
 10  capital-gain    5000 non-null   int64 
 11  capital-loss    5000 non-null   int64 
 12  hours-per-week  5000 non-null   int64 
 13  native-country  4972 non-null   object
 14  income          5000 non-null   object
dtypes: int64(6), object(9)
memory usage: 586.1+ KB


In [9]:

# Missing value analysis
missing = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage': (df.isnull().mean() * 100).round(2)
})

display(missing[missing['Missing Values'] > 0])

# Duplicate analysis
print("Total Duplicate Records:", df.duplicated().sum())

# Total missing values
print("\nTotal Missing Values:", df.isnull().sum().sum())


,Missing Values,Percentage
workclass,97,1.94
occupation,97,1.94
native-country,28,0.56


Total Duplicate Records: 0

Total Missing Values: 222


In [10]:

# Save original descriptive statistics
before_stats = df.describe(include='all')

# Display numerical statistics
display(df.describe())

# Display categorical statistics
display(df.describe(include='object'))


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,5000.000000,5.000000e+03,5000.000000,5000.000000,5000.000000,5000.00000
mean,38.489000,1.883775e+05,10.092000,1058.855000,80.178400,40.48260
std,13.541084,1.056094e+05,2.565785,7547.307877,384.773792,12.59866
min,17.000000,1.930200e+04,1.000000,0.000000,0.000000,1.00000
25%,28.000000,1.162162e+05,9.000000,0.000000,0.000000,40.00000
50%,37.000000,1.776560e+05,10.000000,0.000000,0.000000,40.00000
75%,47.000000,2.357055e+05,12.000000,0.000000,0.000000,45.00000
max,90.000000,1.366120e+06,16.000000,99999.000000,3900.000000,99.00000


,workclass,education,marital-status,occupation,relationship,race,sex,native-country,income
count,4903,5000,5000,4903,5000,5000,5000,4972,5000
unique,8,16,7,15,6,5,2,41,4
top,Private,HS-grad,Married-civ-spouse,Craft-repair,Husband,White,Male,United-States,<=50K
freq,3485,1581,2240,651,2001,4273,3333,4461,2529


In [11]:

# Create a copy of the original dataset
df_clean = df.copy()

# Identify categorical columns
categorical_cols = df_clean.select_dtypes(
    include=['object']
).columns

# Fill missing categorical values
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna("Unknown")

print("Missing values handled successfully!")

# Check missing values again
print("\nMissing Values After Cleaning:")
print(df_clean.isnull().sum())

# Check dataset shape
print("\nDataset Shape:", df_clean.shape)


Missing values handled successfully!

Missing Values After Cleaning:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

Dataset Shape: (5000, 15)


In [12]:

# Remove duplicate records
df_clean = df_clean.drop_duplicates()

print("Duplicate records removed successfully!")

print("Remaining duplicates:", df_clean.duplicated().sum())

print("Final dataset shape:", df_clean.shape)


Duplicate records removed successfully!
Remaining duplicates: 0
Final dataset shape: (5000, 15)


In [14]:

# Identify numerical columns
numerical_cols = df_clean.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

print("Numerical Columns:")
print(numerical_cols)


Numerical Columns:
['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']


In [15]:

for col in numerical_cols:

    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[col] < lower_bound) |
        (df_clean[col] > upper_bound)
    ]

    print(f"\nColumn: {col}")
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Outliers:", len(outliers))



Column: age
Lower Bound: -0.5
Upper Bound: 75.5
Outliers: 35

Column: fnlwgt
Lower Bound: -63017.625
Upper Bound: 414939.375
Outliers: 149

Column: education-num
Lower Bound: 4.5
Upper Bound: 16.5
Outliers: 189

Column: capital-gain
Lower Bound: 0.0
Upper Bound: 0.0
Outliers: 383

Column: capital-loss
Lower Bound: 0.0
Upper Bound: 0.0
Outliers: 217

Column: hours-per-week
Lower Bound: 32.5
Upper Bound: 52.5
Outliers: 1420


In [16]:

# Columns selected for extreme-value capping
cap_columns = [
    'fnlwgt',
    'capital-gain',
    'capital-loss',
    'hours-per-week'
]

# Apply 99th percentile capping
for col in cap_columns:

    upper_limit = df_clean[col].quantile(0.99)

    # Count values above the limit
    capped_count = (df_clean[col] > upper_limit).sum()

    # Cap extreme values
    df_clean[col] = df_clean[col].clip(
        upper=upper_limit
    )

    print(f"{col}:")
    print("Upper limit:", upper_limit)
    print("Values capped:", capped_count)
    print()

print("Outlier treatment completed!")
print("Dataset shape:", df_clean.shape)


fnlwgt:
Upper limit: 498796.8600000003
Values capped: 50

capital-gain:
Upper limit: 15024.0
Values capped: 36

capital-loss:
Upper limit: 1980.0
Values capped: 46

hours-per-week:
Upper limit: 78.02000000000044
Values capped: 50

Outlier treatment completed!
Dataset shape: (5000, 15)


In [17]:

# Check data types
print("Data Types Before Correction:")
print(df_clean.dtypes)

# Check income categories
print("\nIncome Categories:")
print(df_clean['income'].unique())


Data Types Before Correction:
age                 int64
workclass          object
fnlwgt            float64
education          object
education-num       int64
marital-status     object
occupation         object
relationship       object
race               object
sex                object
capital-gain        int64
capital-loss        int64
hours-per-week    float64
native-country     object
income             object
dtype: object

Income Categories:
['<=50K' '<=50K.' '>50K' '>50K.']


In [18]:
categorical_cols = df_clean.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df_clean[col] = df_clean[col].str.strip().str.rstrip('.')

print("Income Categories After Cleaning:")
print(df_clean['income'].unique())

Income Categories After Cleaning:
['<=50K' '>50K']


In [19]:
# Identify numerical columns
numerical_cols = df_clean.select_dtypes(
    include=['int64', 'float64']
).columns

# Ensure numerical columns are numeric
for col in numerical_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Convert categorical columns to category
categorical_cols = df_clean.select_dtypes(
    include=['object']
).columns

for col in categorical_cols:
    df_clean[col] = df_clean[col].astype('category')

print("Data Types After Correction:")
print(df_clean.dtypes)

Data Types After Correction:
age                  int64
workclass         category
fnlwgt             float64
education         category
education-num        int64
marital-status    category
occupation        category
relationship      category
race              category
sex               category
capital-gain         int64
capital-loss         int64
hours-per-week     float64
native-country    category
income            category
dtype: object


In [20]:
df_clean.info()

print("\nMissing Values:")
print(df_clean.isnull().sum())

print("\nDataset Shape:")
print(df_clean.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             5000 non-null   int64   
 1   workclass       5000 non-null   category
 2   fnlwgt          5000 non-null   float64 
 3   education       5000 non-null   category
 4   education-num   5000 non-null   int64   
 5   marital-status  5000 non-null   category
 6   occupation      5000 non-null   category
 7   relationship    5000 non-null   category
 8   race            5000 non-null   category
 9   sex             5000 non-null   category
 10  capital-gain    5000 non-null   int64   
 11  capital-loss    5000 non-null   int64   
 12  hours-per-week  5000 non-null   float64 
 13  native-country  5000 non-null   category
 14  income          5000 non-null   category
dtypes: category(9), float64(2), int64(4)
memory usage: 282.5 KB

Missing Values:
age               0
workclass

In [21]:
# 1. Group people by age
df_clean['age_group'] = pd.cut(
    df_clean['age'],
    bins=[0, 25, 45, 65, float('inf')],
    labels=['Young', 'Adult', 'Middle-aged', 'Senior']
)

# 2. Identify married individuals
df_clean['is_married'] = (
    df_clean['marital-status']
    .astype(str)
    .str.startswith('Married')
    .astype(int)
)

# 3. Identify individuals with capital gain
df_clean['has_capital_gain'] = (
    df_clean['capital-gain'] > 0
).astype(int)

# 4. Identify individuals with capital loss
df_clean['has_capital_loss'] = (
    df_clean['capital-loss'] > 0
).astype(int)

# 5. Categorize weekly working hours
df_clean['work_hours_group'] = pd.cut(
    df_clean['hours-per-week'],
    bins=[-1, 20, 40, 60, float('inf')],
    labels=['Low', 'Normal', 'High', 'Very High']
)

# 6. Identify individuals working overtime
df_clean['works_overtime'] = (
    df_clean['hours-per-week'] > 40
).astype(int)

print("New Features Created Successfully!")
print(df_clean[
    ['age_group', 'is_married', 'has_capital_gain',
     'has_capital_loss', 'work_hours_group', 'works_overtime']
].head())

New Features Created Successfully!
     age_group  is_married  has_capital_gain  has_capital_loss  \
0        Adult           0                 0                 1   
1  Middle-aged           0                 0                 0   
2        Adult           1                 0                 0   
3  Middle-aged           1                 0                 0   
4        Adult           0                 0                 0   

  work_hours_group  works_overtime  
0             High               1  
1           Normal               0  
2           Normal               0  
3        Very High               1  
4           Normal               0  


In [22]:
new_features = [
    'age_group',
    'is_married',
    'has_capital_gain',
    'has_capital_loss',
    'work_hours_group',
    'works_overtime'
]

print("Updated Dataset Shape:", df_clean.shape)

print("\nMissing Values in New Features:")
print(df_clean[new_features].isnull().sum())

print("\nFeature Summary:")
print(df_clean[new_features].describe(include='all'))

Updated Dataset Shape: (5000, 21)

Missing Values in New Features:
age_group           0
is_married          0
has_capital_gain    0
has_capital_loss    0
work_hours_group    0
works_overtime      0
dtype: int64

Feature Summary:
       age_group   is_married  has_capital_gain  has_capital_loss  \
count       5000  5000.000000       5000.000000       5000.000000   
unique         4          NaN               NaN               NaN   
top        Adult          NaN               NaN               NaN   
freq        2552          NaN               NaN               NaN   
mean         NaN     0.460200          0.076600          0.043400   
std          NaN     0.498463          0.265982          0.203776   
min          NaN     0.000000          0.000000          0.000000   
25%          NaN     0.000000          0.000000          0.000000   
50%          NaN     0.000000          0.000000          0.000000   
75%          NaN     1.000000          0.000000          0.000000   
max        

In [23]:
# Identify all categorical columns
categorical_features = df_clean.select_dtypes(
    include=['object', 'category']
).columns.tolist()

# Remove income because it is our target variable
categorical_features.remove('income')

print("Categorical Features to Encode:")
print(categorical_features)

print("\nTotal Categorical Features:", len(categorical_features))

Categorical Features to Encode:
['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'age_group', 'work_hours_group']

Total Categorical Features: 10


In [24]:
# Create a separate copy
df_encoded = df_clean.copy()

# Encode income target
# 0 = Income <=50K
# 1 = Income >50K
df_encoded['income'] = df_encoded['income'].map({
    '<=50K': 0,
    '>50K': 1
})

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(
    df_encoded,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print("Categorical Encoding Completed!")
print("Encoded Dataset Shape:", df_encoded.shape)

display(df_encoded.head())

Categorical Encoding Completed!
Encoded Dataset Shape: (5000, 112)


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,income,is_married,has_capital_gain,has_capital_loss,...,native-country_United-States,native-country_Unknown,native-country_Vietnam,native-country_Yugoslavia,age_group_Adult,age_group_Middle-aged,age_group_Senior,work_hours_group_Normal,work_hours_group_High,work_hours_group_Very High
0,41,53956.0,13,0,1980,56.0,0,0,0,1,...,1,0,0,0,1,0,0,0,1,0
1,46,431515.0,13,0,0,40.0,0,0,0,0,...,1,0,0,0,0,1,0,1,0,0
2,45,185385.0,9,0,0,40.0,0,1,0,0,...,1,0,0,0,1,0,0,1,0,0
3,54,83103.0,13,0,0,67.0,0,1,0,0,...,1,0,0,0,0,1,0,0,0,1
4,38,219902.0,9,0,0,30.0,0,0,0,0,...,1,0,0,0,1,0,0,1,0,0


In [25]:
print("Remaining Categorical Columns:")
print(
    df_encoded.select_dtypes(
        include=['object', 'category']
    ).columns.tolist()
)

print("\nIncome Values:")
print(df_encoded['income'].value_counts())

print("\nTotal Missing Values:")
print(df_encoded.isnull().sum().sum())

Remaining Categorical Columns:
['income']

Income Values:
income
0    3819
1    1181
Name: count, dtype: int64

Total Missing Values:
0


In [26]:
df_encoded['income'] = df_encoded['income'].astype(int)

print("Income Data Type:", df_encoded['income'].dtype)

print("\nRemaining Categorical Columns:")
print(
    df_encoded.select_dtypes(
        include=['object', 'category']
    ).columns.tolist()
)

Income Data Type: int64

Remaining Categorical Columns:
[]


In [27]:
from sklearn.preprocessing import StandardScaler

# Select numerical columns, excluding income (target)
numerical_features = df_encoded.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

numerical_features.remove('income')

print("Numerical Features to Scale:")
print(numerical_features)

print("\nTotal Numerical Features:", len(numerical_features))

Numerical Features to Scale:
['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week', 'is_married', 'has_capital_gain', 'has_capital_loss', 'works_overtime', 'workclass_Federal-gov', 'workclass_Local-gov', 'workclass_Private', 'workclass_Self-emp-inc', 'workclass_Self-emp-not-inc', 'workclass_State-gov', 'workclass_Unknown', 'workclass_Without-pay', 'education_11th', 'education_12th', 'education_1st-4th', 'education_5th-6th', 'education_7th-8th', 'education_9th', 'education_Assoc-acdm', 'education_Assoc-voc', 'education_Bachelors', 'education_Doctorate', 'education_HS-grad', 'education_Masters', 'education_Preschool', 'education_Prof-school', 'education_Some-college', 'marital-status_Married-AF-spouse', 'marital-status_Married-civ-spouse', 'marital-status_Married-spouse-absent', 'marital-status_Never-married', 'marital-status_Separated', 'marital-status_Widowed', 'occupation_Adm-clerical', 'occupation_Armed-Forces', 'occupation_Craft-repair', 'occupation_Exe

In [28]:
from sklearn.preprocessing import StandardScaler

# Select only continuous numerical features
numerical_features = [
    'age',
    'fnlwgt',
    'education-num',
    'capital-gain',
    'capital-loss',
    'hours-per-week'
]

print("Continuous Numerical Features to Scale:")
print(numerical_features)

print("\nTotal Numerical Features:", len(numerical_features))

Continuous Numerical Features to Scale:
['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Total Numerical Features: 6


In [29]:
# Create a separate copy
df_scaled = df_encoded.copy()

# Initialize StandardScaler
scaler = StandardScaler()

# Standardize only continuous numerical features
df_scaled[numerical_features] = scaler.fit_transform(
    df_scaled[numerical_features]
)

print("Feature Scaling Completed!")

display(df_scaled[numerical_features].head())

Feature Scaling Completed!


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
0,0.185454,-1.329496,1.133490,-0.241988,5.195990,1.283963
1,0.554738,2.436462,1.133490,-0.241988,-0.210386,-0.030327
2,0.480881,-0.018558,-0.425643,-0.241988,-0.210386,-0.030327
3,1.145592,-1.038769,1.133490,-0.241988,-0.210386,2.187538
4,-0.036116,0.325731,-0.425643,-0.241988,-0.210386,-0.851759


In [30]:
print("Means After Standardization:")
print(df_scaled[numerical_features].mean().round(3))

print("\nStandard Deviations After Standardization:")
print(df_scaled[numerical_features].std().round(3))

Means After Standardization:
age               0.0
fnlwgt           -0.0
education-num    -0.0
capital-gain      0.0
capital-loss     -0.0
hours-per-week    0.0
dtype: float64

Standard Deviations After Standardization:
age               1.0
fnlwgt            1.0
education-num     1.0
capital-gain      1.0
capital-loss      1.0
hours-per-week    1.0
dtype: float64


In [31]:
binary_features = [
    'is_married',
    'has_capital_gain',
    'has_capital_loss',
    'works_overtime'
]

print("Binary Feature Values After Scaling:")

for col in binary_features:
    print(col, sorted(df_scaled[col].unique()))

Binary Feature Values After Scaling:
is_married [np.int64(0), np.int64(1)]
has_capital_gain [np.int64(0), np.int64(1)]
has_capital_loss [np.int64(0), np.int64(1)]
works_overtime [np.int64(0), np.int64(1)]


In [32]:
print("DESCRIPTIVE STATISTICS BEFORE PREPROCESSING")

before_stats = df.describe(include='all')

display(before_stats)

DESCRIPTIVE STATISTICS BEFORE PREPROCESSING


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
count,5000.000000,4903,5.000000e+03,5000,5000.000000,5000,4903,5000,5000,5000,5000.000000,5000.000000,5000.00000,4972,5000
unique,NaN,8,NaN,16,NaN,7,15,6,5,2,NaN,NaN,NaN,41,4
top,NaN,Private,NaN,HS-grad,NaN,Married-civ-spouse,Craft-repair,Husband,White,Male,NaN,NaN,NaN,United-States,<=50K
freq,NaN,3485,NaN,1581,NaN,2240,651,2001,4273,3333,NaN,NaN,NaN,4461,2529
mean,38.489000,NaN,1.883775e+05,NaN,10.092000,NaN,NaN,NaN,NaN,NaN,1058.855000,80.178400,40.48260,NaN,NaN
std,13.541084,NaN,1.056094e+05,NaN,2.565785,NaN,NaN,NaN,NaN,NaN,7547.307877,384.773792,12.59866,NaN,NaN
min,17.000000,NaN,1.930200e+04,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1.00000,NaN,NaN
25%,28.000000,NaN,1.162162e+05,NaN,9.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.00000,NaN,NaN
50%,37.000000,NaN,1.776560e+05,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.00000,NaN,NaN
75%,47.000000,NaN,2.357055e+05,NaN,12.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,45.00000,NaN,NaN


In [33]:
print("DESCRIPTIVE STATISTICS AFTER PREPROCESSING")

after_stats = df_clean.describe(include='all')

display(after_stats)

DESCRIPTIVE STATISTICS AFTER PREPROCESSING


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,...,capital-loss,hours-per-week,native-country,income,age_group,is_married,has_capital_gain,has_capital_loss,work_hours_group,works_overtime
count,5000.000000,5000,5000.000000,5000,5000.000000,5000,5000,5000,5000,5000,...,5000.000000,5000.000000,5000,5000,5000,5000.000000,5000.000000,5000.000000,5000,5000.000000
unique,NaN,9,NaN,16,NaN,7,16,6,5,2,...,NaN,NaN,42,2,4,NaN,NaN,NaN,4,NaN
top,NaN,Private,NaN,HS-grad,NaN,Married-civ-spouse,Craft-repair,Husband,White,Male,...,NaN,NaN,United-States,<=50K,Adult,NaN,NaN,NaN,Normal,NaN
freq,NaN,3485,NaN,1581,NaN,2240,651,2001,4273,3333,...,NaN,NaN,4461,3819,2552,NaN,NaN,NaN,3063,NaN
mean,38.489000,NaN,187245.596400,NaN,10.092000,NaN,NaN,NaN,NaN,NaN,...,77.050400,40.369200,NaN,NaN,NaN,0.460200,0.076600,0.043400,NaN,0.297000
std,13.541084,NaN,100265.796542,NaN,2.565785,NaN,NaN,NaN,NaN,NaN,...,366.270926,12.175087,NaN,NaN,NaN,0.498463,0.265982,0.203776,NaN,0.456982
min,17.000000,NaN,19302.000000,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,...,0.000000,1.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,0.000000
25%,28.000000,NaN,116216.250000,NaN,9.000000,NaN,NaN,NaN,NaN,NaN,...,0.000000,40.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,0.000000
50%,37.000000,NaN,177656.000000,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,...,0.000000,40.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,0.000000
75%,47.000000,NaN,235705.500000,NaN,12.000000,NaN,NaN,NaN,NaN,NaN,...,0.000000,45.000000,NaN,NaN,NaN,1.000000,0.000000,0.000000,NaN,1.000000


In [34]:
print("NUMERICAL STATISTICS BEFORE PREPROCESSING")
display(df.describe())

print("\nNUMERICAL STATISTICS AFTER PREPROCESSING")

display(
    df_clean.select_dtypes(
        include=['int64', 'float64']
    ).describe()
)

NUMERICAL STATISTICS BEFORE PREPROCESSING


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,5000.000000,5.000000e+03,5000.000000,5000.000000,5000.000000,5000.00000
mean,38.489000,1.883775e+05,10.092000,1058.855000,80.178400,40.48260
std,13.541084,1.056094e+05,2.565785,7547.307877,384.773792,12.59866
min,17.000000,1.930200e+04,1.000000,0.000000,0.000000,1.00000
25%,28.000000,1.162162e+05,9.000000,0.000000,0.000000,40.00000
50%,37.000000,1.776560e+05,10.000000,0.000000,0.000000,40.00000
75%,47.000000,2.357055e+05,12.000000,0.000000,0.000000,45.00000
max,90.000000,1.366120e+06,16.000000,99999.000000,3900.000000,99.00000



NUMERICAL STATISTICS AFTER PREPROCESSING


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,is_married,has_capital_gain,has_capital_loss,works_overtime
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,38.489000,187245.596400,10.092000,597.741600,77.050400,40.369200,0.460200,0.076600,0.043400,0.297000
std,13.541084,100265.796542,2.565785,2470.379093,366.270926,12.175087,0.498463,0.265982,0.203776,0.456982
min,17.000000,19302.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,28.000000,116216.250000,9.000000,0.000000,0.000000,40.000000,0.000000,0.000000,0.000000,0.000000
50%,37.000000,177656.000000,10.000000,0.000000,0.000000,40.000000,0.000000,0.000000,0.000000,0.000000
75%,47.000000,235705.500000,12.000000,0.000000,0.000000,45.000000,1.000000,0.000000,0.000000,1.000000
max,90.000000,498796.860000,16.000000,15024.000000,1980.000000,78.020000,1.000000,1.000000,1.000000,1.000000


In [35]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Separate target and input features
X = df_clean.drop(columns=['income']).copy()

y = df_clean['income'].astype(str).map({
    '<=50K': 0,
    '>50K': 1
})

# Define feature groups
continuous_features = [
    'age', 'fnlwgt', 'education-num',
    'capital-gain', 'capital-loss', 'hours-per-week'
]

binary_features = [
    'is_married',
    'has_capital_gain',
    'has_capital_loss',
    'works_overtime'
]

categorical_features = [
    col for col in X.columns
    if col not in continuous_features + binary_features
]

# Convert categorical columns to string format
X[categorical_features] = X[categorical_features].astype(str)

print("Input Features Shape:", X.shape)
print("Target Shape:", y.shape)
print("Continuous Features:", continuous_features)
print("Binary Features:", binary_features)
print("Categorical Features:", categorical_features)

Input Features Shape: (5000, 20)
Target Shape: (5000,)
Continuous Features: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Binary Features: ['is_married', 'has_capital_gain', 'has_capital_loss', 'works_overtime']
Categorical Features: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'age_group', 'work_hours_group']


In [36]:
# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, continuous_features),
        ('cat', categorical_transformer, categorical_features),
        ('binary', 'passthrough', binary_features)
    ]
)

# Create complete pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

print("Automated preprocessing pipeline created successfully!")

Automated preprocessing pipeline created successfully!


In [37]:
# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Fit pipeline on training data
X_train_processed = pipeline.fit_transform(X_train)

# Transform testing data
X_test_processed = pipeline.transform(X_test)

# Display results
print("Training Data Shape:", X_train_processed.shape)
print("Testing Data Shape:", X_test_processed.shape)

print("\nPipeline execution completed successfully!")

Training Data Shape: (4000, 119)
Testing Data Shape: (1000, 119)

Pipeline execution completed successfully!


In [38]:
print("========== FINAL DATASET SUMMARY ==========")

print("Original Dataset Shape:", df.shape)
print("Cleaned Dataset Shape:", df_clean.shape)
print("Encoded Dataset Shape:", df_encoded.shape)
print("Scaled Dataset Shape:", df_scaled.shape)

print("\nTotal Missing Values:", df_clean.isnull().sum().sum())
print("Total Duplicate Rows:", df_clean.duplicated().sum())

print("\nNewly Created Features:")
print([
    'age_group',
    'is_married',
    'has_capital_gain',
    'has_capital_loss',
    'work_hours_group',
    'works_overtime'
])

print("\nTraining Processed Shape:", X_train_processed.shape)
print("Testing Processed Shape:", X_test_processed.shape)

print("\n========== VERIFICATION COMPLETE ==========")

========== FINAL DATASET SUMMARY ==========
Original Dataset Shape: (5000, 15)
Cleaned Dataset Shape: (5000, 21)
Encoded Dataset Shape: (5000, 112)
Scaled Dataset Shape: (5000, 112)

Total Missing Values: 0
Total Duplicate Rows: 0

Newly Created Features:
['age_group', 'is_married', 'has_capital_gain', 'has_capital_loss', 'work_hours_group', 'works_overtime']

Training Processed Shape: (4000, 119)
Testing Processed Shape: (1000, 119)

========== VERIFICATION COMPLETE ==========


In [39]:
df_clean.to_csv("adult_cleaned.csv", index=False)
df_encoded.to_csv("adult_encoded.csv", index=False)
df_scaled.to_csv("adult_scaled.csv", index=False)

print("All datasets saved successfully!")

All datasets saved successfully!


In [40]:
from google.colab import files

files.download("adult_cleaned.csv")
files.download("adult_encoded.csv")
files.download("adult_scaled.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>